# 3. praktika: lehen LoRA egokitzailea entrenatu

**Helburua**: `Qwen2.5-0.5B-Instruct` LoRA bidez doitu IKT-txartelak sailkatzeko, `peft` + `trl` erabiliz. Egokitzailea Drive-n gordeko dugu (4. eta 5. saioetan behar da).

**GPU behar da** (T4). Entrenamenduak 5–10 minutu inguru behar ditu.

## 1. Ingurunea

In [ ]:
# COLAB: liburutegiak instalatu (bertsio finkoak, unitate osoan berdinak)
!pip install -q transformers==5.22.2 peft==0.21.2 trl==1.17.1 datasets==5.2.0

In [ ]:
import torch
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# COLAB: Google Drive muntatu (emaitzak gordetzeko eta datuak irakurtzeko)
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
# Aldatu zure Drive-ko karpetara. Bertan egon behar dute txartelak_train.jsonl eta txartelak_test.jsonl
KARPETA = "/content/drive/MyDrive/llm-egokitzapena"
os.makedirs(KARPETA, exist_ok=True)
TRAIN_FITX = f"{KARPETA}/txartelak_train.jsonl"
TEST_FITX = f"{KARPETA}/txartelak_test.jsonl"

# Fitxategiak Drive-n ez badaude, igo eskuz
if not os.path.exists(TRAIN_FITX):
    from google.colab import files  # COLAB
    igoak = files.upload()
    for izena in igoak:
        os.replace(izena, f"{KARPETA}/{izena}")
print(os.listdir(KARPETA))

In [ ]:
import json
def irakurri_jsonl(fitx):
    with open(fitx, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

train_lista = irakurri_jsonl(TRAIN_FITX)
test_lista = irakurri_jsonl(TEST_FITX)
SYSTEM = train_lista[0]["prompt"][0]["content"]   # sistema-mezua datuetatik (beti berdina!)
print(len(train_lista), len(test_lista))
print(SYSTEM)

## 2. Eredua eta datuak

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

MODELOA = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODELOA)
# T4-ak ez du bf16 onartzen: fp32-n kargatu eta fp16 mixed precision-ekin entrenatu
model = AutoModelForCausalLM.from_pretrained(MODELOA, dtype=torch.float32, device_map="auto")

train = load_dataset("json", data_files=TRAIN_FITX, split="train")
test = load_dataset("json", data_files=TEST_FITX, split="train")
print(train, test, sep="\n")

## 3. Arkitektura aztertu

Bilatu geruza linealak (`Linear`). Horietan jarriko ditugu LoRA matrizeak.

In [ ]:
print(model)

In [ ]:
# Geruza lineal motak eta haien tamainak (lehen geruzakoak)
for izena, modulua in model.named_modules():
    if isinstance(modulua, torch.nn.Linear) and ".layers.0." in izena:
        print(f"{izena:45} {modulua.in_features:5} -> {modulua.out_features}")
print("Geruza kopurua:", model.config.num_hidden_layers)

### Ariketa: LoRA parametroak eskuz kalkulatu

`Linear(d_in -> d_out)` geruza batek, `r` heinarekin, `r * (d_in + d_out)` parametro gehitzen ditu. Kalkulatu guztira `r=16`-rekin, 7 geruza motetan eta geruza guztietan. Gero `print_trainable_parameters()`-ekin egiaztatu.

In [ ]:
r = 16
guztira = 0
for izena, modulua in model.named_modules():
    if isinstance(modulua, torch.nn.Linear) and izena.split(".")[-1] in \
            ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]:
        guztira += r * (modulua.in_features + modulua.out_features)
print(f"LoRA parametroak: {guztira:,}  ({guztira / model.num_parameters():.2%})")

## 4. Konfigurazioa

In [ ]:
from peft import LoraConfig
from trl import SFTConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

config = SFTConfig(
    output_dir="txartelak-lora",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="no",
    max_length=512,
    fp16=True, bf16=False,          # T4. A100/L4 GPUetan: bf16=True, fp16=False
    report_to="none",
    seed=42,
)

## 5. Trainer-a sortu eta datuak egiaztatu

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=config,
    train_dataset=train,
    eval_dataset=test,
    processing_class=tokenizer,
    peft_config=lora_config,
)
trainer.model.print_trainable_parameters()

In [ ]:
# Zer jasotzen du ereduak? input_ids eta labels (-100 = galeran ez da kontuan hartzen)
b = trainer.train_dataset[0]
print(b.keys())
ids, labels = b["input_ids"], b["labels"]
print("Token guztiak:", len(ids), "| Galeran:", sum(l != -100 for l in labels))
print("\n--- Galeran kontuan hartzen den zatia ---")
print(repr(tokenizer.decode([i for i, l in zip(ids, labels) if l != -100])))

> **Galdera**: `<|im_end|>` agertzen da galeran kontuan hartzen den zatian? Zergatik da garrantzitsua?

## 6. Entrenatu!

In [ ]:
trainer.train()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

log = pd.DataFrame(trainer.state.log_history)
tr = log.dropna(subset=["loss"])
ev = log.dropna(subset=["eval_loss"])
plt.plot(tr["step"], tr["loss"], label="train")
plt.plot(ev["step"], ev["eval_loss"], "o-", label="test")
plt.xlabel("urratsa"); plt.ylabel("galera"); plt.legend(); plt.title("Galera-kurbak"); plt.show()
ev[["epoch", "eval_loss", "eval_mean_token_accuracy"]]

## 7. Probatu

In [ ]:
def sailkatu(testua, eredua=None):
    eredua = eredua or trainer.model
    mezuak = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": testua}]
    sarrera = tokenizer.apply_chat_template(mezuak, add_generation_prompt=True,
                                            return_tensors="pt", return_dict=True).to(eredua.device)
    eredua.eval()
    with torch.no_grad():
        irteera = eredua.generate(**sarrera, max_new_tokens=60, do_sample=False)
    return tokenizer.decode(irteera[0][sarrera["input_ids"].shape[1]:], skip_special_tokens=True)

probak = [
    "Kaixo, B14 gelako proiektorea ez da pizten. Klase osoa geldirik dago.",
    "Egun on, D02 gelako inprimagailuak argi gorria du. Ez dago presarik.",   # D02: inoiz ikusi gabe
    "Pasahitza ahaztu dut berriro...",
    "Orientazio-gelan wifia ez dabil eta bilera bat daukagu orain",           # idazkera berria
]
for p in probak:
    print(p, "\n  ->", sailkatu(p))

In [ ]:
# Oinarrizko eredua (egokitzailea desaktibatuta) alderatzeko
with trainer.model.disable_adapter():
    for p in probak[:2]:
        print(p, "\n  OINARRIA ->", sailkatu(p))

## 8. Egokitzailea gorde

In [ ]:
IRTEERA = f"{KARPETA}/txartelak-lora"
trainer.save_model(IRTEERA)          # egokitzailea + tokenizatzailea
!ls -lh {IRTEERA}

In [ ]:
import json
with open(f"{IRTEERA}/adapter_config.json") as f:
    konf = json.load(f)
print({k: konf[k] for k in ["base_model_name_or_path", "r", "lora_alpha", "target_modules"]})

## 9. Esperimentuak (taldeka banatu)

Talde bakoitzak aldagai **bakar** bat aldatu, berriz entrenatu eta galera-kurbak alderatu. Gorde egokitzailea izen desberdinarekin (adib. `txartelak-lora-r4`).

| Taldea | Aldaketa |
|---|---|
| A | `r=4` vs `r=64` (`lora_alpha = 2*r`) |
| B | `num_train_epochs=1` vs `6` |
| C | `learning_rate=2e-5` vs `1e-3` |
| D | 100 adibide bakarrik: `train.select(range(100))` |

Erantzun: zer gertatzen da galerarekin? Eta `probak` adibideekin? Zein da konfigurazio onena zure ustez, eta zergatik?

> **Kontuz**: berriz entrenatu aurretik, kargatu eredua berriro (2. atala). `SFTTrainer`-ek LoRA geruzak gehitzen dizkio ereduari.

In [ ]:
# Zure esperimentua hemen